# 11 — Enhancement Round 5 (estimator noise + calibration)

Round 5 separates estimator randomness from split randomness (R5A: forest seeds under frozen splits) and settles sigmoid-vs-isotonic calibration (R5B). Descriptive unless pre-committed bars clear.


In [ ]:
import sys, json
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))


In [ ]:
S = json.loads((ROOT/'reports/gap6_r5a.json').read_text())
T = json.loads((ROOT/'reports/gap6_r5b.json').read_text())
rms = [s['random']['mean'] for s in S['per_seed']]
gms = [s['group']['mean'] for s in S['per_seed']]
print('forest-seed random range %.4f (seeds %s)' % (max(rms)-min(rms), [s['forest_seed'] for s in S['per_seed']]))
print('forest-seed group range  %.4f' % (max(gms)-min(gms)))
assert abs((max(rms)-min(rms)) - 0.0056) < 5e-4 and abs((max(gms)-min(gms)) - 0.0067) < 5e-4
tw = T['three_way_fresh_identical_splits']
twm = {k: v for k, v in tw.items() if not k.startswith('ranking')}
print('three-way:', {k: (round(v['random']['mean'], 4), round(v['group']['mean'], 4), round(v['brier_random_OOF'], 4)) for k, v in twm.items()})
print('R5B verdict:', T['verdict'])
assert T['verdict'] == 'REJECT'
print('R5 confirmed: estimator noise ~0.4-0.5x split noise; isotonic rejected (Brier not best)')

## Round-5 outcome

- Forest seeds move means by 0.0056 (random) / 0.0067 (group) under frozen splits — second-order vs split noise (0.0134) but same scale as every challenger delta ever measured. Total uncertainty around 0.8440 is at least ~+-0.007 before fold noise. Seeds are not hyperparameters: nothing selected.
- Isotonic matches sigmoid on accuracy-relevant axes but loses Brier; sigmoid stays the recorded alternative. Threshold landscape described only; accuracy stays @0.5. Recorded as E24–E25.
